In [2]:
# ============================================================================
# 04_ablation.ipynb  —  Divergence tests + 2x2 circularity ablation + agreement
# ----------------------------------------------------------------------------
# Single-cell notebook. All code and comments are in English.
# Figures: seaborn grayscale, no captions, dpi=600, saved as both PNG and PDF.
# Run order: 00 -> 01 -> 02 -> 03 -> 04 -> 05.
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Make src/ importable regardless of the notebook's working directory.
_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)

import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
# NOTE: absolute local paths are never printed (anonymity for double-blind review).

# ----------------------------------------------------------------------------
# PURPOSE
#   (a) Metric-judgment divergence: per perturbation, test whether the judge's
#       qualitative score drops more than the faithfulness metric (both on the
#       common 1-5 scale; faithfulness x5). Wilcoxon signed-rank, paired d_z,
#       Holm-corrected family-wise p-values.
#   (b) 2x2 circularity ablation: generation (gap-given/gap-blind) x judge
#       (informed/blind). Report the judge's drop on P2 and P4 per cell.
#   (c) Inter-judge agreement: Spearman rho and quadratic-weighted kappa.
#
# INPUT   data/scored.parquet
# OUTPUT  results/tables/tab_divergence.csv, tab_ablation_2x2.csv
# ----------------------------------------------------------------------------
from scipy.stats import wilcoxon, spearmanr
from statsmodels.stats.multitest import multipletests

s = pd.read_parquet(require_input(DATA_DIR / "scored.parquet",
                                  hint="Run 03_judge_metric first."))
s["faith5"] = s["faithfulness"] * 5.0

# Headline: informed judge on gap-given generation.
head = s[(s.judge_cond=="informed") & (s.condition=="gap_given")].copy()

def paired(pert):
    p0 = head[(head.pert=="P0")].set_index("case_id")["judge_qual"]
    pk = head[(head.pert==pert) & (head.k.isin([0,3]))].set_index("case_id")["judge_qual"]
    j = pd.concat([p0.rename("p0"), pk.rename("pk")], axis=1).dropna()
    m0 = head[(head.pert=="P0")].set_index("case_id")["faith5"]
    mk = head[(head.pert==pert) & (head.k.isin([0,3]))].set_index("case_id")["faith5"]
    jm = pd.concat([m0.rename("m0"), mk.rename("mk")], axis=1).dropna()
    jdrop = (j.p0 - j.pk); mdrop = (jm.m0 - jm.mk)
    div = jdrop.reindex(jm.index).fillna(0) - mdrop
    try: pj = wilcoxon(j.p0, j.pk).pvalue
    except Exception: pj = np.nan
    try: pdv = wilcoxon(div).pvalue
    except Exception: pdv = np.nan
    dz = jdrop.mean() / (jdrop.std(ddof=1) + 1e-9)
    return dict(pert=pert, judge_drop=round(jdrop.mean(),3),
                metric_drop=round(mdrop.mean(),3),
                divergence=round(div.mean(),3), d_z=round(dz,3),
                p_judge=pj, p_div=pdv)

div_rows = [paired(p) for p in ["P1","P2","P3","P4","P5"]]
divt = pd.DataFrame(div_rows)
# Holm correction over the divergence family
mask = divt["p_div"].notna()
divt.loc[mask, "p_div_holm"] = multipletests(divt.loc[mask,"p_div"], method="holm")[1]
save_table(divt, "tab_divergence")
print(divt.to_string(index=False))

# 2x2 circularity ablation: judge drop on P2 and P4 per (generation, judge) cell
def cell_drop(gen_cond, judge_cond, pert):
    d = s[(s.condition==gen_cond) & (s.judge_cond==judge_cond)]
    p0 = d[d.pert=="P0"].set_index("case_id")["judge_qual"]
    pk = d[d.pert==pert].set_index("case_id")["judge_qual"]
    j = pd.concat([p0.rename("p0"), pk.rename("pk")], axis=1).dropna()
    return round((j.p0 - j.pk).mean(), 3) if len(j) else np.nan

abl = []
for gcond in ("gap_given","gap_blind"):
    for jcond in ("informed","blind"):
        abl.append({"generation":gcond, "judge":jcond,
                    "P2_drop":cell_drop(gcond,jcond,"P2"),
                    "P4_drop":cell_drop(gcond,jcond,"P4")})
ablt = pd.DataFrame(abl)
save_table(ablt, "tab_ablation_2x2")
print(ablt.to_string(index=False))

# Inter-judge agreement (primary informed vs blind as a reproducibility proxy;
# replace 'blind' with a true secondary-model column when available).
a = s[(s.pert=="P0") & (s.condition=="gap_given")]
piv = a.pivot_table(index="case_id", columns="judge_cond", values="judge_qual")
piv = piv.dropna()
if len(piv) > 3 and {"informed","blind"}.issubset(piv.columns):
    rho = spearmanr(piv["informed"], piv["blind"]).correlation
    print(f"Spearman rho (informed vs blind): {rho:.3f}  (n={len(piv)})")


[saved] results\tables\tab_divergence.csv
[saved] results\tables\tab_divergence.tex
pert  judge_drop  metric_drop  divergence   d_z  p_judge    p_div  p_div_holm
  P1        0.05       -0.062       0.112 0.083 0.705457 0.336289    0.989460
  P2        0.65        0.016       0.634 0.550 0.025884 0.329820    0.989460
  P3        0.20        0.821      -0.621 0.224 0.317311 0.213122    0.852490
  P4        2.90        0.512       2.388 2.591 0.000095 0.006431    0.032155
  P5        0.15        0.029       0.121 0.307 0.179712 0.860287    0.989460
[saved] results\tables\tab_ablation_2x2.csv
[saved] results\tables\tab_ablation_2x2.tex
generation    judge  P2_drop  P4_drop
 gap_given informed     0.65     2.90
 gap_given    blind     0.30     2.00
 gap_blind informed     0.25     1.10
 gap_blind    blind     0.30     1.25
Spearman rho (informed vs blind): 0.251  (n=17)
